# **Brain-to-text '25**

This notebook 
**29/12/2025 TODO**

- [x] EDA
- [x] DATA UTILS / LOADERS
  - [x] FOR EDA
  - [x] FOR MODEL TRAINING
  - [ ] FOR MODEL SUBMISSION
- [x] BASELINE MODEL EXECUTION
  - [x] DATA UTILS
  - [x] MODEL DOWNLOADS
  - [x] MODEL LOADING PRE-TRAINED + TRAIN / TEST / VAL
  - [x] COMP SUBMISSION DEMO RUN
- [x] MODEL TUNING PLANS
  - [x] PARALLEL MODEL MY OWN
- [x] SCOPE + DOWNLOAD REPOS / URL / LINKS / RESEARCH PAPERS
- [x] TEST GITHUB MODEL INFERENCE

**NOTES**

- Pretrained model was loaded but did not use Redis + LLM to decode. Instead, the pre-trained GRU Decoder model was used as an encoder for the Ngram Embedding Model built here.


**Evaluation Metric**

- Word Error Rate (WER): how accurate your decoded speech predictions match the ground truth transcripts.


**Dataset Overview**

| Column              | Description                                                                                                             |
| ------------------- | ----------------------------------------------------------------------------------------------------------------------- |
| **neural_features** | The raw neural signal recordings (likely from intracortical electrodes) - this is the input data your model will decode |
| **n_time_steps**    | Number of time steps per trial's neural recording                                                                       |
| **seq_class_ids**   | Character/phoneme class labels for each time step (ground truth sequence for training)                                  |
| **seq_len**         | Length of the target transcription sequence                                                                             |
| **transcriptions**  | The ASCII representation of the intended text/speech - your prediction target                                           |
| **sentence_label**  | Raw text representation of the intended text/speech                                                                     |
| **session**         | Recording session identifier (different days/sessions)                                                                  |
| **block_num**       | Block number within a session (trials are grouped into blocks)                                                          |
| **trial_num**       | Individual trial identifier within a block                                                                              |


## Project Directory Utils


In [ ]:
# CONFIG SETUP
# !pip install --upgrade torch torchvision torchaudio

from pathlib import Path

INPUT_DIR = Path("/kaggle/input/brain-to-text-25")
OUTPUT_DIR = Path("/kaggle/working/")
# Input Comp Dataset
NEURAL_DATA_DIR = INPUT_DIR / "t15_copyTask_neuralData" / "hdf5_data_final"
PRETRAINED_DIR = (
    INPUT_DIR / "t15_pretrained_rnn_baseline" / "t15_pretrained_rnn_baseline"
)
CKPT_DIR = PRETRAINED_DIR / "checkpoint"
ARGS_PATH = CKPT_DIR / "args.yaml"
CKPT_PATH = CKPT_DIR / "best_checkpoint"
# Mounted Github Baseline Model
BASELINE_MODEL_PATH = Path("/kaggle/input/baseline-model/pytorch/default/1")
KB_PATH = OUTPUT_DIR / "knowledgeBase.pkl"
OUTPUT_PATH = OUTPUT_DIR / "submission.csv"

## Data Utils


### **UTILS FOR EDA**


In [ ]:
# DATA UTILS TO PLAY WITH DATASET

import h5py
from typing import Literal


def load_h5py_file(file_path):
    data = {
        "neural_features": [],
        "n_time_steps": [],
        "seq_class_ids": [],
        "seq_len": [],
        "transcriptions": [],
        "sentence_label": [],
        "session": [],
        "block_num": [],
        "trial_num": [],
    }
    # Open the hdf5 file for that day
    with h5py.File(file_path, "r") as f:

        keys = list(f.keys())

        # For each trial in the selected trials in that day
        for key in keys:
            g = f[key]

            neural_features = g["input_features"][:]
            n_time_steps = g.attrs["n_time_steps"]
            seq_class_ids = g["seq_class_ids"][:] if "seq_class_ids" in g else None
            seq_len = g.attrs["seq_len"] if "seq_len" in g.attrs else None
            transcription = g["transcription"][:] if "transcription" in g else None
            sentence_label = (
                g.attrs["sentence_label"][:] if "sentence_label" in g.attrs else None
            )
            session = g.attrs["session"]
            block_num = g.attrs["block_num"]
            trial_num = g.attrs["trial_num"]

            data["neural_features"].append(neural_features)
            data["n_time_steps"].append(n_time_steps)
            data["seq_class_ids"].append(seq_class_ids)
            data["seq_len"].append(seq_len)
            data["transcriptions"].append(transcription)
            data["sentence_label"].append(sentence_label)
            data["session"].append(session)
            data["block_num"].append(block_num)
            data["trial_num"].append(trial_num)
    return data


def load_neural_dataset(
    data_type: Literal["train", "val", "test"] = "train",
    max_load: int | None = MAX_LOAD,
):
    data = []

    for idx, file in enumerate(NEURAL_DATA_DIR.rglob(f"*{data_type}*")):
        if file.is_file():
            z = load_h5py_file(str(file))
            data.append(z)

        if idx and idx == max_load:
            return data

    return data


# train_data = load_neural_dataset("train")
# single_sample = str(list(NEURAL_DATA_DIR.rglob("*test*"))[0])
# z = load_h5py_file(single_sample)
# def gen():
#    yield from pd.DataFrame(z).iterrows()
# for _, i in gen():
#    print(i)
#    break
# def postprocess_neural_features(features, input_layer: int = 0):
# i['neural_features']
#    neural_input = np.expand_dims(features,axis=0).astype(np.float32)
#    inputs = torch.from_numpy(neural_input).to('cpu')
#    logits = runSingleDecodingStep(inputs, input_layer, model, config, 'cpu')
#    return logits

# logits = postprocess_neural_features(i['neural_features'], 0)
# decode_single_item(logits)

### **UTILS FOR VALIDATING EDA**


In [ ]:
import torch
import numpy as np
import pandas as pd

# train_df = pd.DataFrame(train_data)

ID_COLS = ["session", "trial_num", "block_num"]
INPUTS = ["neural_features", "n_time_steps", "seq_class_ids", "seq_len"]
PREDS = ["transcriptions", "sentence_label"]

# Quick testing before building the model training data utils
NEURAL_HIDDEN_DIM = 512


def validate_data_shape(row):
    seq_len = np.array(row["seq_len"])  # (59,)
    n_time_steps = np.array(row["n_time_steps"])  # (59,)
    assert (
        seq_len.shape == n_time_steps.shape
    ), f"Expected same shape for seq_len and n_time_steps. {n_time_steps.shape} & {seq_len.shape}"

    seq_class_ids = np.array(row["seq_class_ids"])  # (59, 500)
    trans = np.array(row["transcriptions"])  # (59, 500)
    assert (
        seq_class_ids.shape == trans.shape
    ), f"Expected same shape for seq_len and n_time_steps. {seq_class_ids.shape} & {trans.shape}"

    neural_features = np.vstack(row["neural_features"])
    assert (
        neural_features.shape[-1] == NEURAL_HIDDEN_DIM
    ), f"Neural features expected shape (*, 512) but got {neural_features.shape}"

    print(
        f"seq_len: {seq_len.shape}\nn_time_steps: {n_time_steps.shape}\nseq_class_ids: {seq_class_ids.shape}\ntranscriptions: {trans.shape}\nneural_features: {neural_features.shape}"
    )


# validate_data_shape(train_df.iloc[-1])

## Pre-trained Model Loader


In [ ]:
# Pretrained Model Loaders

import sys
import yaml
import torch
import importlib.util

DEVICE = "cpu"
NEURAL_HIDDEN_DIM = 512
LOGIT_TO_PHONEME = [
    "BLANK",  # "BLANK" = CTC blank symbol
    "AA",
    "AE",
    "AH",
    "AO",
    "AW",
    "AY",
    "B",
    "CH",
    "D",
    "DH",
    "EH",
    "ER",
    "EY",
    "F",
    "G",
    "HH",
    "IH",
    "IY",
    "JH",
    "K",
    "L",
    "M",
    "N",
    "NG",
    "OW",
    "OY",
    "P",
    "R",
    "S",
    "SH",
    "T",
    "TH",
    "UH",
    "UW",
    "V",
    "W",
    "Y",
    "Z",
    "ZH",
    " | ",  # "|" = silence token
]


def _mount_model_path(baseline_model_path: Path):
    rnn_path = baseline_model_path / "model_training" / "rnn_model.py"

    if not rnn_path.exists():
        raise FileNotFoundError(f"Could not find: {rnn_path}")

    spec = importlib.util.spec_from_file_location("rnn_model", str(rnn_path))

    if spec:
        rnn_model = importlib.util.module_from_spec(spec)
        sys.modules["rnn_model"] = rnn_model
        spec.loader.exec_module(rnn_model)  # type: ignore
        return rnn_model
    else:
        raise ValueError


def load_pretrained(
    args_path: Path = ARGS_PATH,
    checkpoint_path: Path = CKPT_PATH,
    baseline_model_path: Path = BASELINE_MODEL_PATH,
):
    rnn_model = _mount_model_path(baseline_model_path)

    with args_path.open("r") as f:
        cfg = yaml.safe_load(f)

    # top-level model block
    m_cfg = cfg["model"]
    d_cfg = cfg["dataset"]

    neural_dim = d_cfg.get("neural_dim", m_cfg.get("n_input_features"))
    n_units = m_cfg["n_units"]
    n_layers = m_cfg["n_layers"]
    rnn_dropout = m_cfg.get("rnn_dropout", 0.0)
    input_dropout = m_cfg.get("input_network", {}).get("input_layer_dropout", 0.0)
    patch_size = m_cfg.get("patch_size", 0)
    patch_stride = m_cfg.get("patch_stride", 0)
    n_classes = d_cfg["n_classes"]
    n_days = 50

    model = rnn_model.GRUDecoder(
        neural_dim=neural_dim,
        n_units=n_units,
        n_days=n_days,
        n_classes=n_classes,
        rnn_dropout=rnn_dropout,
        input_dropout=input_dropout,
        n_layers=n_layers,
        patch_size=patch_size,
        patch_stride=patch_stride,
    )

    model = model.to(DEVICE)

    # weights_only=False needed for checkpoints with numpy objects (PyTorch 2.6+)
    # dict_keys(['model_state_dict', 'optimizer_state_dict', 'scheduler_state_dict', 'val_PER', 'val_loss'])
    state = torch.load(checkpoint_path, map_location=DEVICE, weights_only=False)
    state_dict = state.get("model_state_dict", None)

    # Fix key prefixes - iterate over a copy to avoid mutating dict during iteration
    new_state_dict = {}
    for key in state_dict.keys():
        new_key = key.replace("module.", "").replace("_orig_mod.", "")
        new_state_dict[new_key] = state_dict[key]

    model.load_state_dict(new_state_dict, strict=False)
    model.to(DEVICE)
    model.eval()

    return model, cfg


# Initiate the pre-trained model
model, config = load_pretrained()

## Eval Metric Utils


In [ ]:
from typing import List, Tuple


def _edit_distance(ref_words: List[str], hyp_words: List[str]) -> int:
    """Word-level Levenshtein distance."""
    n = len(ref_words)
    m = len(hyp_words)
    dp = [[0] * (m + 1) for _ in range(n + 1)]

    for i in range(n + 1):
        dp[i][0] = i  # deletions
    for j in range(m + 1):
        dp[0][j] = j  # insertions

    for i in range(1, n + 1):
        for j in range(1, m + 1):
            if ref_words[i - 1] == hyp_words[j - 1]:
                cost = 0
            else:
                cost = 1
            dp[i][j] = min(
                dp[i - 1][j] + 1,  # deletion
                dp[i][j - 1] + 1,  # insertion
                dp[i - 1][j - 1] + cost,  # substitution
            )
    return dp[n][m]


def brain_to_text_wer(pairs: List[Tuple[str, str]]) -> float:
    """
    Compute aggregate WER over validation samples.

    pairs: list of (true_sentence, pred_sentence) in plain text.
    Returns WER in [0,1].
    """
    total_edit = 0
    total_words = 0

    for ref, hyp in pairs:
        # competition uses word-level comparison after simple whitespace split [web:1][web:23]
        ref_words = ref.strip().split()
        hyp_words = hyp.strip().split()
        if not ref_words:
            continue
        total_edit += _edit_distance(ref_words, hyp_words)
        total_words += len(ref_words)

    if total_words == 0:
        return 0.0

    return total_edit / total_words

## Knowledge Base + Ngram


In [ ]:
# KB Temporary

import pickle
import numpy as np
from collections import Counter, defaultdict
from typing import Iterable, List, Tuple, Dict, Optional

## **NGram**
import re
from pathlib import Path
from sklearn.feature_extraction.text import HashingVectorizer
from sklearn.linear_model import SGDClassifier

CONTEXT_SIZE = 2
TOP_K = 5
N_FEATURES = 2**20
BATCH_SIZE = 256

BOS_TOKEN = BOS = "<s>"
EOS_TOKEN = EOS = "</s>"
PAD_TOKEN = UNK = "<unk>"
TOKEN_PATTERN = re.compile(
    r"(</s>|<s>)"  # match BOS/EOS tags as single tokens
    r"|[A-Za-z']+"  # words + contractions
    r"|[0-9]+"  # numbers
    r"|[^\sA-Za-z0-9]"  # punctuation/symbols
)


def tokenize(text: str):
    return TOKEN_PATTERN.findall(text)


def make_training_pairs(texts, n_context=CONTEXT_SIZE):
    """
    Yields (context_string, next_word) pairs. Optionally maps OOV words to UNK if vocab is provided.
    """

    for t in texts:
        toks = tokenize(t)
        toks = [BOS] * n_context + toks
        print(toks)
        for i in range(n_context, len(toks)):
            ctx = " ".join(toks[i - n_context : i])
            nxt = toks[i]
            yield ctx, nxt


class OnlineNGramClassifier:
    def __init__(self, n_context=CONTEXT_SIZE, n_features=N_FEATURES):
        """
        base_texts: initial corpus to define vocab/classes (true sentences).
        """
        self.n_context = n_context
        self.n_features = n_features

        self.vocab = {BOS, UNK}
        self.vec = HashingVectorizer(
            n_features=self.n_features,
            alternate_sign=False,
            norm="l2",
        )
        self.clf = SGDClassifier(
            loss="log_loss",
            alpha=1e-5,
            learning_rate="optimal",
        )
        self._is_fitted = False

    @property
    def classes(self):
        return np.array(sorted(self.vocab))

    def partial_fit(self, texts: list, batch_size=BATCH_SIZE):
        """
        Incrementally fit on a list (or iterable) of new sentences.
        """
        X_batch, y_batch = [], []

        for ctx, nxt in make_training_pairs(texts, n_context=self.n_context):
            X_batch.append(ctx)
            y_batch.append(nxt)
            # print("ctx", ctx, "->", nxt)
            self.vocab.add(nxt)

            if len(X_batch) >= batch_size:
                X = self.vec.transform(X_batch)
                y = np.array(y_batch)
                self.clf.partial_fit(X, y, classes=self.classes)
                X_batch, y_batch = [], []
                if not self._is_fitted:
                    self._is_fitted = True

        # flush remainder
        if X_batch:
            X = self.vec.transform(X_batch)
            y = np.array(y_batch)
            if not self._is_fitted:
                self.clf.partial_fit(X, y, classes=self.classes)
                self._is_fitted = True
            else:
                self.clf.partial_fit(X, y)

    def predict_next(self, last_words, topk=5):
        """
        last_words: list of previous tokens (e.g. ["bring", "it"])
        returns list of (token, prob).
        """
        if not self._is_fitted:
            raise RuntimeError("Model not fitted yet. Call partial_fit first.")

        toks = [w if w in self.vocab else UNK for w in last_words][-self.n_context :]
        ctx = " ".join(([BOS] * (self.n_context - len(toks))) + toks)
        X = self.vec.transform([ctx])
        proba = self.clf.predict_proba(X)[0]
        idx = np.argsort(proba)[::-1][:topk]
        return [(self.classes[i], float(proba[i])) for i in idx]

    # ---------- persistence ----------
    def save(self, path: str | Path):
        path = Path(path)
        path.parent.mkdir(parents=True, exist_ok=True)
        state = {
            "n_context": self.n_context,
            "n_features": self.n_features,
            "vocab": list(self.vocab),
            "classes": self.classes,
            "clf": self.clf,
            # HashingVectorizer has no learned state, but keep config if needed
        }
        with open(path, "wb") as f:
            pickle.dump(state, f, protocol=pickle.HIGHEST_PROTOCOL)

    @classmethod
    def load(cls, path: str | Path) -> "OnlineNGramClassifier":
        path = Path(path)
        with open(path, "rb") as f:
            state = pickle.load(f)

        # reconstruct with dummy base_texts; we will overwrite attributes
        obj = cls(n_context=state["n_context"], n_features=state["n_features"])
        obj.vocab = set(state["vocab"])
        obj.classes = state["classes"]
        obj.clf = state["clf"]
        obj._is_fitted = True  # loaded model is already trained
        # recreate vectorizer with same settings
        obj.vec = HashingVectorizer(
            n_features=state["n_features"],
            alternate_sign=False,
            norm="l2",
        )
        return obj


class KnowledgeBase:
    def __init__(self, smoothing: float = 1.0):
        self.smoothing = smoothing
        # P(true | phonetic) mapping: raw -> Counter(true)
        self.phonetic2token = defaultdict(Counter)
        # phonetic vocab: counts
        self.phoneticCounts = Counter()
        # P(true_next | true_curr) mapping: true_curr -> Counter(true_next)
        # self.token_bigram = defaultdict(Counter)
        # readable vocab: counts
        # self.token_unigram = Counter()           # true_curr -> total

    def update(self, true_seq: list, pred_seq: list) -> None:
        # Inputs ["hi how are you", "asdfasdfa"] etc.
        true_seqs = [tok for sent in true_seq for tok in sent.split()]
        pred_seqs = [tok for sent in pred_seq for tok in sent.split()]

        # P(true | phonetic) mapping: raw_tok -> true_word
        for t_word, r_tok in zip(true_seqs, pred_seqs):
            self.phonetic2token[r_tok][t_word] += 1
            self.phoneticCounts[r_tok] += 1

        # P(true_next | true_curr): curr_true -> next_word mapping
        # for w_curr, w_next in zip(true_seq, true_seq[1:]):
        #    self.token_bigram[w_curr][w_next] += 1
        #    self.token_unigram[w_curr] += 1
        if not hasattr(self, "ngram_model"):
            setattr(self, "ngram_model", OnlineNGramClassifier())

        self.ngram_model.partial_fit(true_seq)

    def translate(
        self, raw_tok: str, cand_true: Optional[Iterable[str]] = None
    ) -> Dict[str, float]:
        """
        P(true | raw_tok): given a raw/phoneme token, return probabilities over
        readable true tokens.

        If cand_true is None, uses all true words that have ever co-occurred
        with this raw token.
        """
        if raw_tok not in self.phoneticCounts:
            return {}

        true_counts = self.phonetic2token.get(raw_tok)
        total_for_raw = self.phoneticCounts.get(raw_tok)

        if not true_counts or not total_for_raw:
            return {}

        if cand_true is None:
            candidates = list(true_counts.keys())
        else:
            cand_true = list(cand_true)
            candidates = [t for t in cand_true if t in true_counts]

        if not candidates:
            return {}

        vocab_size = max(len(true_counts), 1)
        probs = {}
        for t in candidates:
            c_xy = true_counts[t]
            p = (c_xy + self.smoothing) / (total_for_raw + self.smoothing * vocab_size)
            probs[t] = p

        # normalize
        Z = sum(probs.values())
        if Z > 0:
            for k in probs:
                probs[k] /= Z

        return probs

    def proof_reader(self, translated_sequence: list | np.ndarray):
        assert hasattr(
            self, "ngram_model"
        ), "Non existing ngram_model pls pass initial/base text corpus with `update` method function."
        print(translated_sequence[:1])

        cleaned = []
        for seq in translated_sequence:
            try:
                seq_token = seq.split()

                if all(tok == PAD_TOKEN for tok in seq_token):
                    cleaned.append(seq)
                else:
                    tokens = []
                    for idx, token in enumerate(seq_token):
                        if token == PAD_TOKEN:
                            seq_context_tokens = seq_token[:idx]
                            predictions = self.ngram_model.predict_next(
                                seq_context_tokens
                            )
                            # predict_next returns [(token, prob), ...], extract best token
                            replace_word = predictions[0][0] if predictions else token
                        else:
                            replace_word = token

                        tokens.append(replace_word)

                    cleaned.append(" ".join(tokens))
            except Exception as e:
                print("Inserted:", translated_sequence[0], "-> Tried splitting:", seq)
                raise e
        return cleaned

    # Convenience helpers to persist the KB
    def save(self, path: Path):
        path = Path(path)
        path.parent.mkdir(parents=True, exist_ok=True)
        with open(path, "wb") as f:
            pickle.dump(self, f, protocol=pickle.HIGHEST_PROTOCOL)

    @staticmethod
    def load(path: Path) -> "KnowledgeBase":
        with open(path, "rb") as f:
            kb = pickle.load(f)
        return kb

## Dataset & Dataloaders


In [ ]:
# Data Loaders, Dataset Iterators & Collators For Experimentation

import sys
import h5py
import pickle
from typing import Literal
from tqdm.auto import tqdm
from torch.utils.data import Dataset, DataLoader


def _mount_baseline_modules(baseline_model_path: Path = BASELINE_MODEL_PATH):
    """Add baseline model directories to current running imports."""

    paths_to_add = [
        str(baseline_model_path),
        str(baseline_model_path / "model_training"),
    ]
    for p in paths_to_add:
        if p not in sys.path:
            sys.path.insert(0, p)


# Mount once at import time
_mount_baseline_modules()

from model_training.evaluate_model_helpers import (
    runSingleDecodingStep,
)  # pyright: ignore[reportMissingImports]

MAX_LOAD = None
NEURAL_HIDDEN_DIM = 512
COLS = [
    "neural_features",
    "n_time_steps",
    "seq_class_ids",
    "seq_len",
    "transcriptions",
    "sentence_label",
    "session",
    "block_num",
    "trial_num",
]


def load_h5py_file(file_path):
    data = {
        "neural_features": [],
        "n_time_steps": [],
        "seq_class_ids": [],
        "seq_len": [],
        "transcriptions": [],
        "sentence_label": [],
        "session": [],
        "block_num": [],
        "trial_num": [],
    }
    # Open the hdf5 file for that day
    with h5py.File(file_path, "r") as f:

        keys = list(f.keys())

        # For each trial in the selected trials in that day
        for key in keys:
            g = f[key]

            neural_features = g["input_features"][:]
            n_time_steps = g.attrs["n_time_steps"]
            seq_class_ids = g["seq_class_ids"][:] if "seq_class_ids" in g else None
            seq_len = g.attrs["seq_len"] if "seq_len" in g.attrs else None
            transcription = g["transcription"][:] if "transcription" in g else None
            sentence_label = (
                g.attrs["sentence_label"][:] if "sentence_label" in g.attrs else None
            )
            session = g.attrs["session"]
            block_num = g.attrs["block_num"]
            trial_num = g.attrs["trial_num"]

            data["neural_features"].append(neural_features)
            data["n_time_steps"].append(n_time_steps)
            data["seq_class_ids"].append(seq_class_ids)
            data["seq_len"].append(seq_len)
            data["transcriptions"].append(transcription)
            data["sentence_label"].append(sentence_label)
            data["session"].append(session)
            data["block_num"].append(block_num)
            data["trial_num"].append(trial_num)
    return data


class BrainToTextDataset(Dataset):
    """This is to be used for submission."""

    def __init__(
        self,
        data_type: Literal["train", "test", "val"],
        data_config: dict = config,
        max_load: int | None = MAX_LOAD,
        input_path: Path = NEURAL_DATA_DIR,
        kb_path: Path = KB_PATH,
    ):
        self._type = data_type
        self._max_load = max_load
        self.input_path = input_path
        self.config = data_config
        self.sessions_labels = self.config.get("dataset", {}).get("sessions", [])
        self.sessions = sorted(list(self.input_path.rglob(f"*{self._type}*")))
        self.sessions_to_day_idx = {
            label: idx for idx, label in enumerate(self.sessions_labels)
        }

        # If not train or val, load the knowledge base + ngram models
        # ----------------------------------------------------------------
        # For "test", this dataset will load a pre-trained KnowledgeBase
        # from disk if kb_path is supplied and kb is None.
        # This lets you keep LM and decoder logic consistent across splits. [web:18][web:27]
        # ----------------------------------------------------------------
        # if self._type == "test":
        #    if kb_path is not None and kb_path.exists():
        #        with open(kb_path, "rb") as f:
        #            self.kb = pickle.load(f)
        #    else:
        #        self.kb = None

        print(
            f"This Dataset Class will load {self._type} dataset up to {self._max_load}."
        )

    def __getitem__(self, idx):
        try:
            if self._max_load is not None and idx >= self._max_load:
                raise IndexError

            ss_file = self.sessions[idx]
            print(f"Loading sessions: {ss_file}")
            row = load_h5py_file(str(ss_file))

            # seq_len = np.array(row['seq_len']) # (59,)
            n_time_steps = np.array(row["n_time_steps"])  # (59,)
            # assert seq_len.shape == n_time_steps.shape, f"Expected same shape for seq_len and n_time_steps. {n_time_steps.shape} & {seq_len.shape}"
            # seq_class_ids = np.array(row['seq_class_ids']) # (59, 500)
            # trans = np.array(row['transcriptions']) # (59, 500)
            # assert seq_class_ids.shape == trans.shape, f"Expected same shape for seq_len and n_time_steps. {seq_class_ids.shape} & {trans.shape}"

            input_layer = self.sessions_to_day_idx[str(ss_file.parent.name)]
            # put neural data through the pretrained model to get phoneme predictions (logits)
            neural_features = np.vstack(row["neural_features"])
            assert (
                neural_features.shape[-1] == NEURAL_HIDDEN_DIM
            ), f"Neural features expected shape (*, 512) but got {neural_features.shape}"
            neural_input = np.expand_dims(neural_features, axis=0).astype(
                np.float32
            )  # (1, *, 512)
            features = torch.from_numpy(neural_input).to(DEVICE)
            logits = runSingleDecodingStep(
                features, input_layer, model, self.config, DEVICE
            )
            print(f"Neural Shape: {features.shape} -> Logits shape: {logits.shape}")

            sentence_label = row.get("sentence_label", [])

            if self._type != "test":
                true_label, pred_label = postprocess(
                    true_labels=sentence_label, pred_logits=logits
                )
            else:
                true_label, pred_label = [], postprocess_decoded_logits(logits)

            return {
                # "seq_len": seq_len,
                "n_time_steps": n_time_steps,
                # "seq_class_ids": seq_class_ids,
                # "transcriptions": trans,
                "sentence_label": sentence_label,
                # "neural_features": features,
                "logits": logits,
                "pred_label": pred_label,
                "true_label": true_label,
            }

        except Exception as e:
            raise e

    def __len__(self):
        return (
            min(len(self.sessions), self._max_load)
            if self._max_load is not None
            else len(self.sessions)
        )


def decode_single_item(logits: np.ndarray):
    # Decodes single session
    # assert logits.shape[0] == 1 and len(logits.shape) == 3
    print(f"[decode_single_item]: {logits.shape}")
    pred_seq = (
        logits.squeeze(0).argmax(axis=-1)
        if len(logits.shape) >= 2
        else logits.argmax(-1)
    )
    # remove blanks
    pred_seq = [int(p) for p in pred_seq if p != 0]
    # remove consecutive duplicates and zeros
    pred_seq = [
        pred_seq[i]
        for i in range(len(pred_seq))
        if i == 0 or pred_seq[i] != pred_seq[i - 1]
    ]
    # map to text
    pred_seq = [LOGIT_TO_PHONEME[p] for p in pred_seq]
    # print(f"\nPreview Pred Sequence: {''.join(pred_seq[:50])}")
    return pred_seq


def preprocess_sentence_labels(sentence_labels: list):
    # Returns sequences trials ["hi how are you?", "", "cool im ok."]

    SENTENCE_DELIM = "\n\n"
    return (
        SENTENCE_DELIM.join(sentence_labels)
        .translate(str.maketrans("", "", ",?'\""))
        .split("\n")
    )


def postprocess_decoded_logits(logits: np.ndarray):
    # Returns list of completed tokens
    sample = decode_single_item(logits)
    tok_str = "".join(sample)
    return tok_str.split(" | ")


def postprocess(true_labels: list, pred_logits: np.ndarray):
    # indents the pred sentence parsers accordingly

    true_tokens = preprocess_sentence_labels(true_labels)
    pred_tokens = postprocess_decoded_logits(pred_logits)

    idx = 0
    true_tokens_, pred_tokens_ = [], []
    for trial in true_tokens:

        true_seq = trial.split()
        seq_len = len(true_seq)

        if seq_len > 0:

            pred_seq = pred_tokens[idx : idx + seq_len]
            idx += seq_len
            # retains the fullstop
            if true_seq[-1].endswith("."):
                replace_token = pred_seq[-1] + "."
                pred_seq[-1] = replace_token

            true_seq += [EOS_TOKEN]
            true_tokens_.append(" ".join([BOS_TOKEN] + true_seq))
            pred_tokens_.append(" ".join([BOS_TOKEN] + pred_seq + [EOS_TOKEN]))

    # print(f"\nInput true token sequence: {len(true_labels)}")
    # print(f"Preprocess true token sequence: {len(true_tokens)}")
    # print(f"Postprocess pred logits {len(pred_tokens)}")
    # print(f"[NEW] Postprocess pred logits {len(pred_tokens_)}")
    # true_tokens_ = [i.lower() for i in true_tokens if i != ""]
    assert len(pred_tokens_) == len(true_labels) and len(true_tokens_) == len(
        true_labels
    ), f"{len(pred_tokens_)} != {len(true_tokens)} [pred : true]"

    return true_tokens_, pred_tokens_


def run_trainer(
    smoothing: float = 0.85,
    kb_path: Path = KB_PATH,
    max_load=MAX_LOAD,
    save: bool = False,
):
    data = BrainToTextDataset(data_type="train", max_load=max_load)
    kb = KnowledgeBase(smoothing=smoothing)

    input_data = []

    for item in tqdm(
        data, desc="Loading training dataset", unit="trial", leave=True
    ):  # pyright: ignore[reportCallIssue, reportArgumentType]
        true_label, pred_label = item["true_label"], item["pred_label"]
        kb.update(true_label, pred_label)
        input_data.append(item)

    if kb_path is not None and save is True:
        kb.save(kb_path)

    return kb, input_data


def run_submission(
    kb_path: Path = KB_PATH,
    kb: None | KnowledgeBase = None,
    output_csv: Path = OUTPUT_PATH,
    max_load: int | None = None,
):
    """
    1. Uses KnowledgeBase.translate to produce sentence text (placeholder: simple per-token argmax translate for now)
    2. Runs NextWord Prediction Model (Ngram) to normalize the decoded sequences
    2. Writes submission CSV with id,text. [web:18][web:23]
    """

    if kb is None:
        kb = KnowledgeBase.load(kb_path)

    data = BrainToTextDataset(data_type="test", max_load=max_load)

    all_sentences = []
    all_ids = []
    idx_counter = 0
    for item in tqdm(data, desc="Decoding test", unit="session", leave=True):  # type: ignore
        pred_label: List[str] = item["pred_label"]

        translated_tokens = [
            (
                max(kb.translate(j).items(), key=lambda x: x[1])[0]
                if kb.translate(j)
                else j
            )
            for seq in pred_label
            for j in seq.split()
        ]

        pred_sentence_segments: List[str] = []
        for seq_text in " ".join(translated_tokens).split(BOS_TOKEN):
            seq_text = seq_text.strip().strip(EOS_TOKEN).strip()
            if seq_text:
                pred_sentence_segments.append(seq_text)

        cleaned_list = kb.proof_reader(translated_sequence=pred_sentence_segments)
        cleaned_sentence = " ".join(cleaned_list)

        all_sentences.append(cleaned_sentence)
        all_ids.append(idx_counter)
        idx_counter += 1

    import pandas as pd

    df_out = pd.DataFrame({"id": all_ids, "text": all_sentences})
    output_csv.parent.mkdir(parents=True, exist_ok=True)
    df_out.to_csv(output_csv, index=False)


def run_validation(
    kb_path: Path = KB_PATH,
    kb: None | KnowledgeBase = None,
    max_load: int | None = None,
):
    """Validate KB on val split by comparing decoded sentences to labels.

    Returns a list of (true_sentence, pred_sentence) for quick inspection.
    """
    if kb is None:
        kb = KnowledgeBase.load(kb_path)

    data = BrainToTextDataset(
        data_type="val",
        max_load=max_load,
    )

    pairs = []
    for item in tqdm(data, desc="Validating", unit="session", leave=True):  # type: ignore
        sentence_labels = item["sentence_label"]  # list of original sentences
        pred_label = item["pred_label"]

        pred_sentence_ = []
        for seq_text in " ".join(
            [
                (
                    max(kb.translate(j).items(), key=lambda x: x[1])[0]
                    if kb.translate(j)
                    else j
                )
                for i in pred_label
                for j in i.split()
            ]
        ).split(BOS_TOKEN):
            if seq_text != "":
                pred_sentence_ += [seq_text.strip().strip(EOS_TOKEN).strip()]  # type: ignore

        # Use original label concatenation for quick comparison.
        # true_sentence = " ".join(sentence_labels)
        # Normalize pred_sentence
        pred_sentence = kb.proof_reader(translated_sequence=pred_sentence_)
        pairs.append((sentence_labels, pred_sentence))

    return pairs


kb, data = run_trainer()
# kb, data = run_trainer()

# Saves to Kaggle's working directory

# TEST_SESSION_IDX = 1
# kb, input_data = run_trainer()
# true_sentence_labels, pred_sentence_labels = input_data[TEST_SESSION_IDX]["true_label"], input_data[TEST_SESSION_IDX]["pred_label"]
# PHONETIC_HINTS_ = {j[0]: [j[-1]] for true, pred in zip(true_sentence_labels, pred_sentence_labels) for j in zip(true.split(), pred.split())}

In [192]:
data[0].keys()

dict_keys(['n_time_steps', 'sentence_label', 'logits', 'pred_label', 'true_label'])

In [ ]:
data[0]["pred_label"][:2], data[0]["true_label"][:2]

(['<s> BRIHNG IHT KLOWSER. </s>', '<s> MAY FAEMAHLIY IHZ KLOWSER. </s>'],
 ['<s> Bring it closer. </s>', '<s> My family is closer. </s>'])

In [ ]:
list(zip(data[0]["pred_label"][1], data[0]["true_label"][1]))

[('<', '<'),
 ('s', 's'),
 ('>', '>'),
 (' ', ' '),
 ('M', 'M'),
 ('A', 'y'),
 ('Y', ' '),
 (' ', 'f'),
 ('F', 'a'),
 ('A', 'm'),
 ('E', 'i'),
 ('M', 'l'),
 ('A', 'y'),
 ('H', ' '),
 ('L', 'i'),
 ('I', 's'),
 ('Y', ' '),
 (' ', 'c'),
 ('I', 'l'),
 ('H', 'o'),
 ('Z', 's'),
 (' ', 'e'),
 ('K', 'r'),
 ('L', '.'),
 ('O', ' '),
 ('W', '<'),
 ('S', '/'),
 ('E', 's'),
 ('R', '>')]

In [244]:
results = run_validation(kb=kb, max_load=5)
wer_avg = 0
for idx, (true, pred) in enumerate(results, start=1):
    wer = brain_to_text_wer(zip(true, pred))
    wer_avg += wer
    print("WER AVG:", wer_avg)

print("FINAL WER AVG:", wer_avg / idx)

This Dataset Class will load val dataset up to 5.


Validating:   0%|          | 0/5 [00:00<?, ?session/s]

Loading sessions: /kaggle/input/brain-to-text-25/t15_copyTask_neuralData/hdf5_data_final/t15.2023.08.13/data_val.hdf5


/usr/local/lib/python3.11/dist-packages/torch/amp/autocast_mode.py:266: UserWarning: User provided device_type of 'cuda', but CUDA is not available. Disabling
  warnings.warn(


Neural Shape: torch.Size([1, 25607, 512]) -> Logits shape: (1, 6397, 41)
[decode_single_item]: (1, 6397, 41)
['you can see the code at this point as will.']
Loading sessions: /kaggle/input/brain-to-text-25/t15_copyTask_neuralData/hdf5_data_final/t15.2023.08.18/data_val.hdf5
Neural Shape: torch.Size([1, 34764, 512]) -> Logits shape: (1, 8686, 41)
[decode_single_item]: (1, 8686, 41)
['They RIYJHAHNKLIY RIHLUWKT him.']
Loading sessions: /kaggle/input/brain-to-text-25/t15_copyTask_neuralData/hdf5_data_final/t15.2023.08.20/data_val.hdf5
Neural Shape: torch.Size([1, 35018, 512]) -> Logits shape: (1, 8750, 41)
[decode_single_item]: (1, 8750, 41)
['do you go by the adds when you look at them']
Loading sessions: /kaggle/input/brain-to-text-25/t15_copyTask_neuralData/hdf5_data_final/t15.2023.08.25/data_val.hdf5
Neural Shape: torch.Size([1, 19107, 512]) -> Logits shape: (1, 4772, 41)
[decode_single_item]: (1, 4772, 41)
['LAOMIHKERZ past a measure last year.']
Loading sessions: /kaggle/input/brain

In [190]:
kb.phonetic2token

defaultdict(collections.Counter,
            {'<s>': Counter({'<s>': 8072}),
             'BRIHNG': Counter({'Bring': 17, 'bring': 25}),
             'IHT': Counter({'it': 525,
                      'It': 297,
                      'notice': 1,
                      'believe': 1,
                      'likes': 1,
                      'on': 1,
                      'got': 2,
                      'think': 2,
                      'pick': 1,
                      'Is': 1,
                      'was': 1,
                      'to': 2,
                      'possibly': 1,
                      'I': 2,
                      'just': 1,
                      'rented': 1,
                      'decide': 2,
                      'compare': 1,
                      'like': 1,
                      'do': 1,
                      'did': 1,
                      'house': 1,
                      'of': 1,
                      'make': 3,
                      'take': 1,
                      'said'

## BIGRAM


In [ ]:
import re
import math
from collections import Counter, defaultdict
import torch


# ---------------------------
# 1) Tokenization + vocab
# ---------------------------
def tokenize(text: str):
    # simple word tokenizer: letters, digits, apostrophes
    return re.findall(r"[a-z0-9']+", text.lower())


def build_vocab(corpus_texts, min_count=1):
    counts = Counter()
    for t in corpus_texts:
        counts.update(tokenize(t))
    vocab = [w for w, c in counts.items() if c >= min_count]
    vocab = ["<s>", "</s>"] + sorted(set(vocab) - {"<s>", "</s>"})
    w2i = {w: i for i, w in enumerate(vocab)}
    return vocab, w2i, counts


# ---------------------------
# 2) Bigram (n-gram) LM
# ---------------------------

EOS_TOKEN = "</s>"
BOS_TOKEN = "<s>"


class BigramLM:
    def __init__(self, corpus_texts, vocab, w2i, add_k=0.5):
        self.vocab = vocab
        self.w2i = w2i
        self.V = len(vocab)
        self.add_k = add_k

        # bigram counts: prev -> Counter(next)
        self.bigram = defaultdict(Counter)
        self.unigram = Counter()

        for t in corpus_texts:
            words = [BOS_TOKEN] + tokenize(t) + [EOS_TOKEN]
            self.unigram.update(words)
            for a, b in zip(words, words[1:]):
                self.bigram[a][b] += 1

    def logp(self, prev_word: str, word: str) -> float:
        # add-k smoothing: (c(prev,word)+k)/(c(prev)+k*V)
        c_prev = self.unigram.get(prev_word, 0)
        c_big = self.bigram[prev_word].get(word, 0)
        num = c_big + self.add_k
        den = c_prev + self.add_k * self.V
        return math.log(num / den)


# ---------------------------
# 3) Character n-gram embeddings (hashing trick) in PyTorch
# ---------------------------
def char_ngrams(s: str, n_min=2, n_max=4):
    s = f"^{s}$"
    grams = []
    for n in range(n_min, n_max + 1):
        for i in range(len(s) - n + 1):
            grams.append(s[i : i + n])
    return grams


def hashed_ngram_vec(s: str, D=4096, n_min=2, n_max=4, device="cpu"):
    v = torch.zeros(D, device=device)
    for g in char_ngrams(s.lower(), n_min=n_min, n_max=n_max):
        idx = hash(g) % D
        v[idx] += 1.0
    # normalize so cosine similarity is meaningful
    v = v / (v.norm(p=2) + 1e-8)
    return v


# ---------------------------
# 4) Candidate generation (token -> top-K words)
# ---------------------------
def build_vocab_matrix(vocab, D=4096, device="cpu"):
    M = torch.stack(
        [hashed_ngram_vec(w, D=D, device=device) for w in vocab], dim=0
    )  # [V, D]
    return M


# Optional tiny phonetic-ish hints (add yours as needed)
PHONETIC_HINTS = PHONETIC_HINTS_


def topk_candidates(token, vocab, w2i, vocabM, K=25, device="cpu"):
    token = token.lower()
    # similarity via dot product of normalized vectors (cosine)
    tv = hashed_ngram_vec(token, device=device)
    sims = vocabM @ tv  # [V]

    # take top-K indices
    topk = torch.topk(sims, k=min(K, sims.numel())).indices.tolist()
    cands = [vocab[i] for i in topk if vocab[i] not in (BOS_TOKEN, EOS_TOKEN)]

    # inject hint candidates at the front (if present)
    if token in PHONETIC_HINTS:
        hinted = [w for w in PHONETIC_HINTS[token] if w in w2i]
        # keep unique, hints first
        cands = hinted + [w for w in cands if w not in set(hinted)]

    return cands[:K], sims  # sims returned mainly for debugging


# ---------------------------
# 5) Viterbi decode (Bigram LM + emission similarity)
# ---------------------------
def decode_pred_labels(
    pred_labels: str, corpus_texts, K=25, D=4096, alpha=6.0, device="cpu"
):
    """
    alpha scales how strongly we trust emission similarity vs LM.
    Larger alpha => hug the token shape harder.
    """
    tokens = tokenize(pred_labels)

    vocab, w2i, _ = build_vocab(corpus_texts, min_count=1)
    lm = BigramLM(corpus_texts, vocab, w2i, add_k=K)
    vocabM = build_vocab_matrix(vocab, D=D, device=device)

    # Build candidate sets and emission scores
    cand_lists = []
    emit_scores = []  # list of torch tensors, each [Ct]
    for tok in tokens:
        cands, _ = topk_candidates(tok, vocab, w2i, vocabM, K=K, device=device)
        cand_lists.append(cands)

        # emission = alpha * cosine_similarity(token, cand_word)
        tv = hashed_ngram_vec(tok, D=D, device=device)
        idxs = torch.tensor([w2i[w] for w in cands], device=device)
        sims = vocabM[idxs] @ tv  # [Ct]
        emit_scores.append(alpha * sims)

    # Viterbi DP
    T = len(tokens)
    dp = []
    bp = []  # backpointers

    # init from <s>
    c0 = cand_lists[0]
    dp0 = torch.tensor([lm.logp("<s>", w) for w in c0], device=device) + emit_scores[0]
    dp.append(dp0)
    bp.append(torch.full((len(c0),), -1, dtype=torch.long, device=device))

    # recurrence
    for t in range(1, T):
        prev_c = cand_lists[t - 1]
        cur_c = cand_lists[t]
        prev_dp = dp[t - 1]

        cur_scores = torch.empty(len(cur_c), device=device)
        cur_bp = torch.empty(len(cur_c), dtype=torch.long, device=device)

        for j, w in enumerate(cur_c):
            trans = torch.tensor(
                [lm.logp(prev_w, w) for prev_w in prev_c], device=device
            )  # [Ct-1]
            total = prev_dp + trans
            best_i = torch.argmax(total)
            cur_scores[j] = total[best_i] + emit_scores[t][j]
            cur_bp[j] = best_i

        dp.append(cur_scores)
        bp.append(cur_bp)

    # end transition to </s>
    last_c = cand_lists[-1]
    end_trans = torch.tensor([lm.logp(w, "</s>") for w in last_c], device=device)
    last_scores = dp[-1] + end_trans
    best_last = torch.argmax(last_scores).item()

    # backtrack
    out = []
    j = best_last
    for t in reversed(range(T)):
        out.append(cand_lists[t][j])
        j = bp[t][j].item()
        if j < 0 and t != 0:
            break
    out.reverse()
    return " ".join(out)


TEST_IDX = 1
pred = pred_sentence_labels[TEST_IDX]
corpus = true_sentence_labels
decoded = decode_pred_labels(pred, corpus_texts=corpus, K=1, alpha=0.5)
print(
    f"[INPUT]: {pred}\n[PRED]: {decoded}\n[CORRECT]: {true_sentence_labels[TEST_IDX]}"
)

[INPUT]: ay hhaed ah nayntiyn sehvahntiy eyt verzhahn bihfaor dhihs wahn.
[PRED]: day had a until seventy eight very or his war
[CORRECT]: i had a nineteen seventy eight version before this one.
